# Utilizing a Vector Database for our RAG application

In this notebook, we continue developing our Retrieval-Augmented Generation (RAG) pipeline using Haystack, with a focus on improving document quality and retrieval scalability.

We begin by initializing a Milvus vector database as our document store, enabling persistent and efficient storage of embeddings. A key addition in this notebook is the Document Cleaner component, which enhances text quality by removing extra whitespaces, empty lines, repeated headers/footers, and unwanted substrings or patterns using regex—making the documents more suitable for downstream processing by LLMs.

As in the previous notebook, we split documents into chunks and store them in Milvus. We then test the retrieval process to ensure relevant documents are returned for a given query. Finally, we build and evaluate a complete RAG pipeline that includes a prompt template, query embedder, retriever, and LLM—validating the end-to-end system.

## Setup Environment

### Import packages

In [ ]:
from pathlib import Path
import os
from haystack import Pipeline
from haystack.components.converters import DOCXToDocument
from haystack.components.preprocessors import DocumentCleaner
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.writers import DocumentWriter
from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder

from milvus_haystack import MilvusDocumentStore
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
import textwrap
from dotenv import dotenv_values
from haystack.utils import Secret

#### Helper Functions

In [ ]:
def format_output(llm_reponse:dict)-> str: 
    formatted_text = llm_reponse["llm"]["replies"][0].text
    wrapped_text = "\n".join(
        textwrap.fill(line, width=120, subsequent_indent="  ") if line.strip() else line
        for line in formatted_text.splitlines()
    )

    return wrapped_text

#### Setup VLLM Connection Parameters

In [ ]:
llm_config = dotenv_values('/nvme/scratch/edu29/llm.env')
embd_config = dotenv_values('/nvme/scratch/edu29/embd.env')

### Initialize the Vector Database

Set up the Milvus vector database to store document embeddings. The `drop_old` parameter ensures any existing data is cleared.

In [ ]:
HOME_DIR = os.path.expanduser("~")
db_path = os.path.join(HOME_DIR, "rag-training", "rag_vectordb.db")

In [ ]:
connection_args={"uri": db_path}
document_store = MilvusDocumentStore(
    connection_args=connection_args,
    drop_old=True,
)

## Indexing Documents and performing RAG

Create a pipeline to process, clean, split, embed, and store the documents in the vector database.

### Setup Indexing components

To be able to use our documents we need to perform 5 things:

1. Turn them into compatible Haystack *Documents*.
2. Clean each Document using Haystack's `DocumentCleaner`. This removes any whitespaces, empty lines, specified substrings, regexes and so on.
3. Then we split our documents into *smaller chunks*. We can define various split methods and length.
4. Turn them into embeddings with an *embedder*.
5. Store them in a Haystack *Document Store* so they can be accessed later on.

In [ ]:
DOCUMENTS_DIR = Path("./dummy_data/documents_dir")
FILES = [file.resolve() for file in DOCUMENTS_DIR.rglob("*") if file.is_file()]

In [ ]:
doc_embedder = OpenAIDocumentEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)
splitter = DocumentSplitter(split_by="word", split_length=15)

In [ ]:
# Initialize the indexing pipeline
indexing_pipeline = Pipeline()

# Add each component to the pipeline
indexing_pipeline.add_component("converter", DOCXToDocument())
indexing_pipeline.add_component("cleaner", DocumentCleaner())
indexing_pipeline.add_component("splitter", splitter)
indexing_pipeline.add_component("embedder", doc_embedder)
indexing_pipeline.add_component("writer", DocumentWriter(document_store))

# Connect each component
indexing_pipeline.connect("converter", "cleaner")
indexing_pipeline.connect("cleaner", "splitter")
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.connect("embedder", "writer")

# Run the Pipeline
indexing_pipeline.run({"converter": {"sources": FILES}})

### Testing the retrieval using a Vector Database

In this cell below, we can see the output of our retriever based on our query.

The two components we need is the embedder that turns the query into an embedding, and then the retriever.

In [ ]:
dummy_text_embedder = OpenAITextEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

In [ ]:
question = "Tell me a bit about QuantumStream"  # You can replace it with your own question.

retrieval_pipeline = Pipeline()
retrieval_pipeline.add_component("embedder", dummy_text_embedder)
retrieval_pipeline.add_component("retriever", MilvusEmbeddingRetriever(document_store=document_store, top_k=3))
retrieval_pipeline.connect("embedder", "retriever")

retrieval_results = retrieval_pipeline.run({"embedder": {"text": question}})

for doc in retrieval_results["retriever"]["documents"]:
    print(doc.content)
    print("-" * 10)

### Initialize RAG

#### Prompt Template for user

This prompt template will be used by our LLM to generate a response based on our Query.

Specifically, the LLM will read this text from top to bottom:

- It will read the task, which is to respond to the user's query using the **provided context**.
- It will then read some **General Guidelines**.
- Then it will read the **provided context**.
- And finally it will read the **user's query**.

You can see that we pass the **context** and **user's query** through this template. This is why we use a prompt builder later on. This component constructs prompts dynamically by processing chat messages.

Specifically, the *ChatPromptBuilder* component creates prompts using static or dynamic templates written in Jinja2 syntax, by processing a list of chat messages. The templates contain placeholders like {{ variable }} that are filled with values provided during runtime. You can use it for static prompts set at initialization or change the templates and variables dynamically while running.

In [ ]:
with open("/nvme/scratch/edu29/dummy_data/RAG_TEMPLATE.txt") as rag_file:
    rag_template = rag_file.read()

user_prompt_template = ChatMessage.from_user(rag_template)

#### RAG Components

For the actual RAG pipeline we have the following components:

- The *text_embedder* which takes the user's query and turns it into embeddings
- The *retriever* which retrieves the relevant documents. The retriever is **different** now. We are using one that is compatible with our Vector Database.
- The *chat_generator* which is our LLM
- The *promt_builder* which was explained above.

In [ ]:
chat_generator = OpenAIChatGenerator(api_key=Secret.from_token(llm_config['VLLM_API_KEY']),
                                     api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
                                     model=llm_config['MODEL_NAME'],
)

text_embedder = OpenAITextEmbedder(api_key=Secret.from_token(embd_config['VLLM_API_KEY']),
                                     api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
                                     model=embd_config['MODEL_NAME'],)

In [ ]:
# Initialize RAG pipeline
rag_pipeline = Pipeline()
rag_pipeline.add_component("text_embedder", text_embedder)
rag_pipeline.add_component("retriever", MilvusEmbeddingRetriever(document_store=document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(variables=["query", "documents"], required_variables=["query", "documents"]))
rag_pipeline.add_component("llm", chat_generator)

# Connect the input/output of each component
rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

#### Perform RAG on our data

Feel free to change the question to something else.

Our documents contain information about the following topics:

- Annual Hackathon the company is organizing
- Cybersecurity Awareness Month
- Employee Recognition Program
- New Office Layout Plan
- Office layout redesign plan
- Product X Launch Timeline
- Product Y Launch Timeline
- QuantumStream product CLI Usage
- QuantumStream product Data Encryption feature
- QuantumStream product Plugin System
- QuantumStream product REST API documentation
- QuantumStream product Scheduler feature
- QuantumStream product Scheduling tasks

---

Feel free to ask anything relating to these topics.

**Suggested prompts:**

- "Whats the purpose of the new office layout? Are we loosing our desks??"
- "I am a new employee at the company. Onboard me about the QuantumStream product."
- "I cannot find the relevant email about the Hackathon, can you tell me more details about it?"

In [ ]:
question = "I am a new employee at the company. Onboard me about the QuantumStream product." # Feel free to change this question

response = rag_pipeline.run(
    data={
        "text_embedder": {"text": question}, 
        "prompt_builder": {"template": [user_prompt_template], "query": question}})
formatted_text = response["llm"]["replies"][0].text
wrapped_text = "\n".join(
    textwrap.fill(line, width=120, subsequent_indent="  ") if line.strip() else line
    for line in formatted_text.splitlines()
)

print(wrapped_text)